# 00 · Setup and environment check

Every check below should pass before you work in any other notebook. If one fails, fix it here — the troubleshooting section of `CONTRIBUTING.md` covers the usual causes.

| | |
|---|---|
| **Owner** | Member 1 |
| **Writes** | nothing |
| **Run it** | first, on every new machine or Colab session |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# On a Colab runtime (in the browser, or from VS Code via the Colab extension) it
# clones REPO at BRANCH, installs it, and on every re-run pulls what you pushed since.
# Locally the package is already installed, so this is skipped. Guide: COLAB.md
import importlib.util
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess

    def _secret(name, default):  # your Colab Secret if you set one, else the default
        try:
            from google.colab import userdata

            return userdata.get(name) or default
        except Exception:  # not set, or unreadable here (VS Code cannot read Secrets)
            return default

    # The code to run is REPO @ BRANCH. Set your own fork / branch once, in Colab Secrets,
    # as A03_REPO / A03_BRANCH. In VS Code, edit the defaults below for the session and
    # put them back before committing. See COLAB.md.
    REPO = _secret("A03_REPO", "ThejithaR/EN3150-Assignment-03-CNN")
    BRANCH = _secret("A03_BRANCH", "main")
    REPO_DIR = "/content/en3150-a03"

    def _git(*args):  # never prompts: a failure shows as an error, not a hang
        subprocess.run(["git", *args], check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})

    url = f"https://github.com/{REPO}.git"
    if not os.path.exists(REPO_DIR):
        _git("clone", "--quiet", "--branch", BRANCH, url, REPO_DIR)
    else:  # re-running this cell picks up everything pushed since
        _git("-C", REPO_DIR, "remote", "set-url", "origin", url)
        _git("-C", REPO_DIR, "fetch", "--quiet", "--prune", "origin")
        _git("-C", REPO_DIR, "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    _git("log", "--oneline", "-1")  # prints the exact commit this run uses

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
import platform
from pathlib import Path

## 1 · Python and packages

Expect Python ≥ 3.10 and `torch` to import. `cuda available: False` is fine — every model trains on CPU too.

In [ ]:
import torch
import torchvision

print(f"python      {platform.python_version()}  ({platform.system()})")
print(f"torch       {torch.__version__}")
print(f"torchvision {torchvision.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")

## 2 · The `edgecnn` package

`installed from this clone: True` means the editable install points at this folder, so your edits take effect immediately.

In [ ]:
import edgecnn

print(f"edgecnn {edgecnn.__version__}")
print(f"repository: {paths.REPO_ROOT.name}")
print(f"installed from this clone: {Path(edgecnn.__file__).resolve().is_relative_to(paths.REPO_ROOT)}")

## 3 · Configs

Every experiment should load in every mode, and only `official` should be official.

In [ ]:
for config_file in sorted((paths.CONFIGS_DIR / "experiments").glob("*.yaml")):
    row = {mode: load_config(config_file, mode=mode).is_official for mode in ("synthetic", "debug", "official")}
    print(f"{config_file.stem:<24} official in: {[mode for mode, ok in row.items() if ok]}")

## 4 · Device *(Member 1 — Phase 0)*

This string is recorded with every result, so check it names your hardware.

In [ ]:
from edgecnn.utils import describe_device, resolve_device

print(describe_device(resolve_device("auto")))

## 5 · Contract tests

Green means every interface between members is intact. Skips are expected until the matching stubs are implemented.

In [ ]:
import subprocess

run = subprocess.run(  # pyproject.toml already adds -q
    [sys.executable, "-m", "pytest", "tests/contracts"],
    cwd=paths.REPO_ROOT, capture_output=True, text=True,
)
print(run.stdout.strip().splitlines()[-1] if run.stdout.strip() else run.stderr)

## Next

Open your own section notebook. Until the real split is committed, work with `MODE = "synthetic"`.